# DMDU workshop: Mumbai October 2026
___

### Staff

* Prof. Dr. Om Prakash Damani, Indian Institute of Technology Bombay
* Prof. Dr. Jayendran Venkateswaran, Indian Institute of Technology Bombay
* Dr. Ir. Patrick Steinmann, Delft University of Technology
* Ir. Coen van den Elshout, Delft University of Technology

### Introduction

Dear reader,

This workshop introduces Decision Making under Deep Uncertainty (DMDU) as a challenge, and Exploratory Modelling and Analysis (EMA) as a solution, to using System Dynamics for analysing and supporting policy processes in complex, dynamic, and multi-actor systems. Participants will explore how System Dynamics models can be used not primarily for prediction, but as exploratory tools to study patterns, examine plausible futures, identify vulnerabilities of candidate policies, and compare robust and adaptive strategies across uncertainty. The workshop combines a concise introduction to key DMDU concepts with hands-on exercises in which participants apply a variety of computational techniques to their own System Dynamics models to explore model behaviour and interpret the resulting patterns to derive decision-relevant insights.

To ensure smooth sailing at the start of the workshop pay attention the following points:

* This notebook contains all the instructions and material for the workshop. The notebook is in chronological order following the workshop (see the table below).
* To make you already familiar with the main concepts of DMDU before the workshop we have selected some of the foundational material of the field. We recommend reading these before the start of the workshop. These papers can be found in the workshop map.
* Download, save and extract the workshop map locally on your computer.
* Install Python and the EMA workbench (see the instructions below).

Looking forward to meet you all!

_Coen_


## Day 0

<a id="technical-preparation"></a>

### Installing the EMA workbench

Exploratory Modeling and Analysis (EMA) is a research methodology that uses computational experiments to analyze complex and uncertain systems (Bankes, 1993). That is, exploratory modeling aims at offering computational decision support for decision making under deep uncertainty and Robust Decision Making.

The EMA workbench aims at providing support for performing exploratory modeling with models developed in various modelling packages and environments. Currently, the workbench offers connectors to Vensim , Netlogo, and Excel. The EMA workbench offers support for designing experiments, performing the experiments - including support for parallel processing on both a single machine as well as on clusters-, and analysing the results.

Most of you wont have access to the Vensim DSS version, which is needed to connect your own models to EMA. The analysis and data generation will be done with predefined python models (in this notebook) and datasets (within the workshop map). However, a demonstration will be giving showing how the connector works with DSS and how to translate small SD models into python if no DSS version is available.

![Flu model diagram](flu-diagram-enhanced.png)
This is the Vensim structure of the model

Lets install the workbench.
From version 2.5.0 the workbench requires Python 3.9 or higher. Make sure to update the python kernel of this notebook to Version 3.9 before installing the workbench.


> **ADDED — not part of the original notebook.**
> The original notebook has been split in two so that each part runs on the EMA Workbench version its code was written for. No original code cell has been modified.
>
> - **Part 1 (this notebook)** — Day 0 setup, open exploration, sensitivity analysis and scenario discovery. Runs on **EMA Workbench 3.0.0**.
> - **Part 2** — Policy analysis. Runs on **EMA Workbench 2.5.3**.
>
> Run the ADDED install cell below **first**, before any other cell.

In [ ]:
# Installs the version this part was written for. Run this cell first, before any other cell.
%pip install -q "ema_workbench[recommended]==3.0.0"
import ema_workbench
assert ema_workbench.__version__ == "3.0.0", (
    f"Found {ema_workbench.__version__}. Restart the session (Runtime > Restart session) and run this cell first."
)
print("ADDED CHECK: EMA Workbench", ema_workbench.__version__)

In [ ]:
try:
    import ema_workbench
    print(f"EMA Workbench {ema_workbench.__version__} is installed and ready.")
except ImportError:
    print("EMA Workbench is not installed. Run: %pip install ema_workbench")
    %pip install -U "ema_workbench[recommended]"
    import ema_workbench
    print(f"EMA Workbench {ema_workbench.__version__} is installed and ready.")

## Day 1

<a id="introduction-to-dmdu"></a>

### 1. Introduction to Decision Making Under Deep Uncertainty

<a id="open-exploration"></a>






## The model: Mexican Flu

During this model we will use a venpy model to conduct our analyses. This model is derived from an SD model built by Pruyt & Hamarat (2010) for a conference paper at the ISDC. We have chosen to use an adjusted Venpy model as most of the workshop participants do not have a Vensim DSS licence to connect vensim to EMA and venpy is native in python. The base version of the model can be found at: https://emaworkbench.readthedocs.io/en/latest/examples/example_flu.html.



The paper by Pruyt & Hamarat (2010) presents a small exploratory System Dynamics model related to the dynamics of the 2009 flu pandemic, also known as the Mexican flu, swine flu, or A(H1N1)v. The model was developed in May 2009 in order to quickly foster understanding about the possible dynamics of this new flu variant and to perform rough-cut policy explorations. Later, the model was also used to further develop and illustrate Exploratory Modelling and Analysis.
## Mexican Flu: the basic model

In the first days, weeks and months after the first reports about the outbreak of a new flu variant in Mexico and the USA, much remained unknown about the possible dynamics and consequences of the at the time plausible/imminent epidemic/pandemic of the new flu variant, first known as Swine or Mexican flu and known today as Influenza A(H1N1)v.

The exploratory model presented here is small, simple, high-level, data-poor (no complex/special structures nor detailed data beyond crude guestimates), and history-poor.

The modelled world is divided in three regions: the Western World, the densely populated Developing World, and the scarcely populated Developing World. Only the two first regions are included in the model because it is assumed that the scarcely populated regions are causally less important for dynamics of flu pandemics. Below, the figure shows the basic stock-and-flow structure; its a classic S.I.E.R structure tracking (diseased) population. For a more elaborate description of the model, see Pruyt & Hamarat (2010).

![Flu model diagram](flu-diagram-enhanced.png)

Import all libraries

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from ema_workbench.analysis import lines
from scipy.stats import gaussian_kde
from IPython.display import display
from ema_workbench import Model, RealParameter, TimeSeriesOutcome, SequentialEvaluator, perform_experiments
from ema_workbench.analysis.plotting import kde_over_time
from ema_workbench.analysis import Density
from SALib.sample import sobol as sobol_sample
from SALib.analyze import sobol, pawn
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, balanced_accuracy_score, silhouette_score

Set seed and number of runs

### Choose the outcome and scalar metric
Default: **infected population in region 1**, using its **peak** for sensitivity and threshold discovery.
All ten stocks in both regions are available through OUTCOME_NAMES. Change OUTCOME, then rerun this settings cell and all subsequent cells. All plots, Sobol, PAWN, convergence, threshold discovery, early-peak discovery and clustering use that setting.
METRIC selects peak, final, or time integral for sensitivity and threshold discovery. Final-state KDE always uses the final time. Clustering uses full trajectories; early-peak discovery uses the stock maximum.
For cumulative deceased or recovered stocks, final is usually useful. Monotonic susceptible/immune trajectories may peak initially; choose final or integral if peak is uninformative. Adjust OUTCOME_THRESHOLD to match the selected metric and units.
The ensemble stores all ten stocks. The structured Sobol design is rerun for the chosen scalar metric; its outputs cannot be substituted from the exploratory sample.


In [ ]:
SEED = 42
NR_EXPERIMENTS = 1000
SOBOL_N = 256  # power of two; actual simulations = N * (17 + 2)

# Change these settings, then rerun this cell and all analysis cells below it.
OUTCOME = "infected_population_region_1"
METRIC = "peak"  # "peak", "final", or "integral"
OUTCOME_NAMES = [f"{stock}_population_region_{region}"
                 for stock in ["susceptible", "infected", "recovered", "immune", "deceased"]
                 for region in [1, 2]]
assert OUTCOME in OUTCOME_NAMES
assert METRIC in {"peak", "final", "integral"}
OUTCOME_LABEL = OUTCOME.replace("_", " ").capitalize()
METRIC_LABEL = f"{METRIC.capitalize()} of {OUTCOME_LABEL.lower()}"
DISPLAY_SCALE = 1e6
AXIS_LABEL = f"{OUTCOME_LABEL} (millions of people)"
# Teaching thresholds, not policy recommendations. Adjust to the chosen outcome/metric.
OUTCOME_THRESHOLD = 1_000_000
EARLY_TIME = 15
MIN_PEAK_POPULATION = 10_000

def summarize_trajectory(values, times):
    values = np.asarray(values, dtype=float)
    if METRIC == "peak":
        return np.max(values, axis=-1)
    if METRIC == "final":
        return values[..., -1]
    # Trapezoidal integral, compatible with NumPy versions before and after 2.0.
    return np.sum((values[..., 1:] + values[..., :-1]) * .5 * np.diff(times), axis=-1)

Check path

In [ ]:
MODEL_PATH = Path.cwd() / 'models' / 'Flu model.py'
assert MODEL_PATH.is_file(), f'Cannot find {MODEL_PATH}; change MODEL_PATH if needed.'
spec = importlib.util.spec_from_file_location('flu_module', MODEL_PATH)
flu_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = flu_module
spec.loader.exec_module(flu_module)
flu_model = flu_module.flu_model

In [ ]:
# (code, descriptive name, lower bound, upper bound)
PARAMETERS = [
    ('x11', 'Seasonal immunity R1', 0, .5),
    ('x12', 'Seasonal immunity R2', 0, .5),
    ('x21', 'Fatality rate R1', .0001, .1),
    ('x22', 'Fatality rate R2', .0001, .1),
    ('x31', 'Initial immune fraction R1', 0, .5),
    ('x32', 'Initial immune fraction R2', 0, .5),
    ('x41', 'Interregional contact rate', 0, .9),
    ('x51', 'Permanent immunity R1', 0, .5),
    ('x52', 'Permanent immunity R2', 0, .5),
    ('x61', 'Recovery time R1', .1, .8),
    ('x62', 'Recovery time R2', .1, .8),
    ('x81', 'Root contact rate R1', 1, 10),
    ('x82', 'Root contact rate R2', 1, 10),
    ('x91', 'Infection rate R1', 0, .1),
    ('x92', 'Infection rate R2', 0, .1),
    ('x101', 'Normal contact rate R1', 0, 200),
    ('x102', 'Normal contact rate R2', 0, 200),
]
names = [p[0] for p in PARAMETERS]
labels = {p[0]: f'{p[0]}: {p[1]}' for p in PARAMETERS}
problem = {'num_vars': len(names), 'names': names,
           'bounds': [[p[2], p[3]] for p in PARAMETERS]}
model = Model('mexicanFlu', function=flu_model)
model.uncertainties = [RealParameter(p[0], p[2], p[3]) for p in PARAMETERS]

Set model outcomes

In [ ]:
# Record all ten stocks so plots can switch outcomes without rerunning the ensemble.
model.outcomes = [TimeSeriesOutcome("TIME")] + [TimeSeriesOutcome(n) for n in OUTCOME_NAMES]
# Check one valid parameter combination before launching the ensemble.
probe = flu_model(**{p[0]: (p[2] + p[3]) / 2 for p in PARAMETERS})
missing = set(OUTCOME_NAMES) - set(probe)
assert not missing, f"Model is missing stock histories: {missing}. Use the supplied all-stocks model."

## Inspecting the model

In [ ]:
table = pd.DataFrame(
    PARAMETERS,
    columns=["Code", "Parameter", "Minimum", "Maximum"],
)
print(table.to_string(index=False, col_space=2))

**Student task:** Predict three influential inputs before running the model. Explain the mechanisms and why the assumed ranges matter.

## Open exploration: run the ensemble
This uses a separate exploratory sample. Do not use this sample for Sobol indices.

In [ ]:
SEED = 42
np.random.seed(SEED)
with SequentialEvaluator(model) as evaluator:
    experiments, outcomes = perform_experiments(model, scenarios=NR_EXPERIMENTS, evaluator=evaluator)

In [ ]:
# Shared data for EVERY analysis below. Rerun after changing OUTCOME or METRIC.
time = np.asarray(outcomes["TIME"][0], dtype=float)
trajectories = np.asarray(outcomes[OUTCOME], dtype=float)
assert np.isfinite(trajectories).all(), "Investigate non-finite model output."
assert np.allclose(outcomes["TIME"], time), "Runs must share a time grid."
assert np.all(np.diff(time) > 0)
X = experiments.loc[:, names].astype(float).reset_index(drop=True)
metric_values = summarize_trajectory(trajectories, time)
assert np.std(metric_values) > 0, "The selected metric is constant: choose another metric."
peak_time = time[np.argmax(trajectories, axis=1)]
peak_population = trajectories.max(axis=1)

def safe_kde(values, grid):
    values = np.asarray(values, dtype=float)
    if np.ptp(values) <= 1e-12 * max(1., abs(values.mean())):
        return None
    return gaussian_kde(values)(grid)

print("Selected:", OUTCOME_LABEL, "| Sensitivity/scenario metric:", METRIC)
print("Integral units are people × model-time units; peak/final units are people.")
display(pd.Series(metric_values, name=METRIC_LABEL).describe())


### 1. Time series

In [ ]:
RUN = 2
fig, axes = lines(experiments, outcomes, outcomes_to_show=OUTCOME,
                  experiments_to_show=[RUN], titles={OUTCOME: f"{OUTCOME_LABEL} — run {RUN + 1}"})
axes[OUTCOME].set(xlabel="Model time", ylabel="Number of people")
plt.show()

**Student task:** Change the run number a couple of times. Identify two distinct trajectory patterns. What mechanisms could explain their timing and magnitude?

In [ ]:
fig, axes = lines(
    experiments,
    outcomes,
    outcomes_to_show=OUTCOME,
    titles={OUTCOME: OUTCOME_LABEL},
)

ax = axes[OUTCOME]

for line in ax.lines:
    line.set_color("steelblue")
    line.set_alpha(0.3)
    line.set_linewidth(0.8)

ax.set(xlabel="Model time", ylabel="Number of people")
plt.show()

### 2. Kernel density of end states
A KDE smooths the empirical distribution. It can extend outside physical bounds; the plot shows only nonnegative population values.

In [ ]:
from ema_workbench.analysis import Density

fig, axes = lines(
    experiments,
    outcomes,
    outcomes_to_show=OUTCOME,
    titles={OUTCOME: OUTCOME_LABEL},
    density=Density.KDE,
)

ax = axes[OUTCOME]

for line in ax.lines:
    line.set_color("steelblue")
    line.set_alpha(0.3)
    line.set_linewidth(0.8)

ax.set(xlabel="Model time", ylabel="Number of people")
plt.show()

In [ ]:
fig, axes = lines(
    experiments,
    outcomes,
    outcomes_to_show="deceased_population_region_1",
    titles={"deceased_population_region_1": "Deceased population in region 1"},
    density=Density.KDE,
)

ax = axes["deceased_population_region_1"]
for line in ax.lines:
    line.set_color("steelblue")
    line.set_alpha(0.3)
    line.set_linewidth(0.8)
ax.set_xlabel("Time")
ax.set_ylabel("Number of deceased people")

plt.show()

**Student task:** What are you seeing from the KDE, what is the difference between the two KDE? discuss

### 3. Kernel density at every time step
Each column is a KDE over runs at one model time. All 6,145 time points are evaluated. Identical columns, including the initial condition, are masked because their distribution is a point mass. Density is shown on a log scale so early concentration does not dominate.

In [ ]:
fig, axes = kde_over_time(
    experiments,
    outcomes,
    outcomes_to_show="infected_population_region_1",
    log=True,
)
plt.show()

**Student task:** What are we seeing? Discuss

### 4. Time series colored by one input

In [ ]:
COLOR_PARAMETER = 'x21'  # change to any input code
segments = np.stack([np.column_stack([time, run / 1e6]) for run in trajectories])
fig, ax = plt.subplots(figsize=(10, 5))
collection = LineCollection(segments, cmap='viridis', linewidths=.6, alpha=.5)
collection.set_array(X[COLOR_PARAMETER].to_numpy())
ax.add_collection(collection); ax.autoscale()
fig.colorbar(collection, ax=ax, label=labels[COLOR_PARAMETER])
ax.set(xlabel='Model time', ylabel=AXIS_LABEL)
plt.show()

**Student task:** Compare coloring by x21 and x101. Is the relationship monotonic? Why does a color pattern alone not establish an isolated causal effect?

## Global sensitivity analysis
### 5. Sobol indices for selected scalar metric
Sobol needs its own structured design. Independent uniform inputs are assumed. S1 measures a first-order variance contribution; ST includes interactions. Second-order indices are not computed. Never shuffle or remove design rows.

In [ ]:


from IPython.display import display
from ema_workbench import (
    Model,
    ScalarOutcome,
    SequentialEvaluator,
    perform_experiments,
)
from ema_workbench.em_framework import get_SALib_problem
from ema_workbench.em_framework.evaluators import Samplers
from SALib.analyze import sobol, pawn

# Use the same selected outcome and metric throughout steps 5–10.
ANALYSIS_OUTCOME = OUTCOME
ANALYSIS_METRIC = METRIC
OUTCOME_LABEL = ANALYSIS_OUTCOME.replace("_", " ").capitalize()
METRIC_LABEL = f"{ANALYSIS_METRIC.capitalize()}: {OUTCOME_LABEL}"

def summarize(values, times):
    values = np.asarray(values, dtype=float)
    times = np.asarray(times, dtype=float)

    if ANALYSIS_METRIC == "peak":
        return np.max(values, axis=-1)
    if ANALYSIS_METRIC == "final":
        return values[..., -1]
    if ANALYSIS_METRIC == "integral":
        return np.sum(
            0.5 * (values[..., 1:] + values[..., :-1])
            * np.diff(times),
            axis=-1,
        )
    raise ValueError("METRIC must be 'peak', 'final', or 'integral'.")

# Prepare the existing exploratory ensemble.
time = np.asarray(outcomes["TIME"][0], dtype=float)
trajectories = np.asarray(outcomes[ANALYSIS_OUTCOME], dtype=float)

assert np.isfinite(trajectories).all()
assert np.allclose(outcomes["TIME"], time)
assert np.all(np.diff(time) > 0)

problem = get_SALib_problem(model.uncertainties)
names = problem["names"]
X = experiments.loc[:, names].astype(float).reset_index(drop=True)
metric_values = summarize(trajectories, time)

# Return only the selected scalar, avoiding storage of Sobol trajectories.
def sensitivity_function(**parameters):
    result = flu_model(**parameters)
    value = summarize(result[ANALYSIS_OUTCOME], result["TIME"])
    return {"selected_metric": float(value)}

sensitivity_model = Model("fluSensitivity", function=sensitivity_function)
sensitivity_model.uncertainties = list(model.uncertainties)
sensitivity_model.constants = list(model.constants)
sensitivity_model.outcomes = [ScalarOutcome("selected_metric")]

SOBOL_N = 256  # Power of two; increase for stronger convergence checks.
assert SOBOL_N >= 2 and SOBOL_N & (SOBOL_N - 1) == 0

print(f"Model evaluations: {SOBOL_N * (len(names) + 2):,}")

np.random.seed(SEED)

with SequentialEvaluator(sensitivity_model) as evaluator:
    sobol_experiments, sobol_outcomes = perform_experiments(
        sensitivity_model,
        scenarios=SOBOL_N,
        evaluator=evaluator,
        uncertainty_sampling=Samplers.SOBOL,
        uncertainty_sampling_kwargs={"calc_second_order": False},
    )

sobol_Y = np.asarray(sobol_outcomes["selected_metric"], dtype=float)

assert np.isfinite(sobol_Y).all(), "Investigate invalid runs; do not remove rows."
assert np.std(sobol_Y) > 0, "Selected metric is constant."

Si = sobol.analyze(
    problem,
    sobol_Y,
    calc_second_order=False,
    seed=SEED,
)

sobol_table = pd.DataFrame(
    {key: Si[key] for key in ["S1", "ST", "S1_conf", "ST_conf"]},
    index=names,
)

display(sobol_table.sort_values("ST", ascending=False))

order = np.argsort(Si["ST"])
positions = np.arange(len(names))

fig, ax = plt.subplots(figsize=(9, 7))
ax.errorbar(
    Si["S1"][order], positions - 0.12,
    xerr=Si["S1_conf"][order], fmt="o", label="S1",
)
ax.errorbar(
    Si["ST"][order], positions + 0.12,
    xerr=Si["ST_conf"][order], fmt="o", label="ST",
)
ax.set_yticks(positions, [names[i] for i in order])
ax.set(title=METRIC_LABEL, xlabel="Sobol sensitivity index")
ax.legend()
plt.tight_layout()
plt.show()

**Student task:** Compare S1 and ST for the top three inputs. What do their gaps suggest? Explain why negative estimates or ST above one should trigger convergence checks rather than be interpreted literally.

### 6. PAWN sensitivity
PAWN uses distribution changes rather than variance decomposition. This uses the exploratory ensemble, conditioning each input on ten intervals.

In [ ]:
assert np.std(metric_values) > 0, "Selected metric is constant."

PAWN_INTERVALS = 10

pawn_result = pawn.analyze(
    problem,
    X.to_numpy(),
    metric_values,
    S=PAWN_INTERVALS,
    seed=SEED,
)

pawn_table = pd.DataFrame(
    {
        key: pawn_result[key]
        for key in ["minimum", "median", "maximum", "CV"]
    },
    index=names,
)

display(pawn_table.sort_values("median", ascending=False))

fig, ax = plt.subplots(figsize=(9, 7))
pawn_table["median"].sort_values().plot.barh(ax=ax)
ax.set(
    title=METRIC_LABEL,
    xlabel="Median PAWN index",
    ylabel="Input parameter",
)
plt.tight_layout()
plt.show()

**Student task:** Compare PAWN and Sobol rankings. Explain why they may disagree. Repeat PAWN with S=5 and S=20 and discuss the number of observations per conditioning interval.

### 7. Convergence
Use complete nested Sobol blocks from the same scrambled design; do not take arbitrary random subsets. The shaded intervals are bootstrap uncertainty estimates, not a guarantee of convergence. PAWN uses increasing exploratory subsets and is an indicative stability check, not an independent replicate study.

In [ ]:
block_size = len(names) + 2

base_sizes = [
    2**power
    for power in range(4, int(np.log2(SOBOL_N)) + 1)
]
if SOBOL_N not in base_sizes:
    base_sizes.append(SOBOL_N)

convergence_rows = []

for n in base_sizes:
    estimate = sobol.analyze(
        problem,
        sobol_Y[:n * block_size],
        calc_second_order=False,
        seed=SEED,
    )

    for i, name in enumerate(names):
        convergence_rows.append({
            "N": n,
            "parameter": name,
            "ST": estimate["ST"][i],
            "CI": estimate["ST_conf"][i],
        })

convergence = pd.DataFrame(convergence_rows)

fig, ax = plt.subplots(figsize=(10, 5))

for name in sobol_table.nlargest(5, "ST").index:
    selected = convergence[convergence["parameter"] == name]
    n = selected["N"].to_numpy()
    st = selected["ST"].to_numpy()
    ci = selected["CI"].to_numpy()

    ax.plot(n, st, "o-", label=name)
    ax.fill_between(n, st - ci, st + ci, alpha=0.12)

ax.set(
    title=f"Sobol convergence — {METRIC_LABEL}",
    xlabel=f"Base N; actual simulations = N × {block_size}",
    ylabel="Total-order index ST",
    xscale="log",
)
ax.legend()
plt.tight_layout()
plt.show()

# PAWN stability using increasing exploratory sample sizes.
sample_sizes = sorted({
    min(n, len(X))
    for n in [100, 250, 500, 1000, len(X)]
})

pawn_estimates = [
    pawn.analyze(
        problem,
        X.iloc[:n].to_numpy(),
        metric_values[:n],
        S=PAWN_INTERVALS,
        seed=SEED,
    )
    for n in sample_sizes
]

fig, ax = plt.subplots(figsize=(10, 5))

for name in pawn_table.nlargest(5, "median").index:
    i = names.index(name)
    ax.plot(
        sample_sizes,
        [estimate["median"][i] for estimate in pawn_estimates],
        "o-",
        label=name,
    )

ax.set(
    title=f"PAWN stability — {METRIC_LABEL}",
    xlabel="Number of exploratory experiments",
    ylabel="Median PAWN index",
)
ax.legend()
plt.tight_layout()
plt.show()

**Student task:** Define a convergence criterion before increasing N: for example, stable top-three rankings and sufficiently narrow confidence intervals. Are those conditions met? Repeat with another seed before making a strong claim.

## Scenario discovery
A shallow classification tree induces readable input rules. Held-out scores assess prediction of the chosen event; they do not establish causality. Split the inputs before fitting rules. Adjust an event definition only with a stated rationale, not to improve its test score.

In [ ]:
from ema_workbench.analysis import prim

OUTCOME_THRESHOLD = 1_000_000  # Adjust for the selected outcome and metric.

def run_prim(target, title):
    target = np.asarray(target, dtype=bool)

    print(title)
    print(f"Cases of interest: {target.sum()} / {len(target)}")
    print(f"Prevalence: {target.mean():.1%}")

    if target.all() or not target.any():
        print("PRIM needs both cases of interest and other cases.")
        return None, None

    search = prim.Prim(
        X,
        target,
        peel_alpha=0.05,
        paste_alpha=0.05,
        mass_min=0.05,
    )

    box = search.find_box()

    if box is None:
        print("No box found.")
        return search, None

    box.show_tradeoff()
    plt.show()

    # Display the currently selected box and its input restrictions.
    box.inspect()
    box.inspect(style="graph")
    plt.show()

    return search, box

above_threshold = metric_values > OUTCOME_THRESHOLD

threshold_prim, threshold_box = run_prim(
    above_threshold,
    f"{METRIC_LABEL} > {OUTCOME_THRESHOLD:,}",
)



### 8. Threshold criterion
The example threshold is a teaching choice, not a policy recommendation.

In [ ]:
# Choose an available index shown in the trade-off plot.
threshold_box.inspect(5)
threshold_box.select(5)

**Student task:** Justify a outcome threshold. Explain one rule in model terms. Interpret precision as rule reliability and recall as event coverage. How do prevalence and the threshold affect the findings?

### 10. Clustering trajectories + rule induction
Split runs first. Fit K-means on training outcome trajectories, then assign test trajectories to fixed training centroids. A tree learns input rules for these labels. Curves are sampled at 100 common time points and divided by one shared scale; magnitude and timing both matter. Per-run normalization would ask a different question.

In [ ]:
from ema_workbench.analysis import clusterer, cart, lines
from ema_workbench.analysis.scenario_discovery_util import RuleInductionType

N_CLUSTERS = 3

# Full trajectories for the selected outcome.
data = np.asarray(outcomes[ANALYSIS_OUTCOME], dtype=float)

assert np.isfinite(data).all()
assert 2 <= N_CLUSTERS < len(data)

# CID requires nonzero trajectory complexity.
complexity = np.sqrt(np.sum(np.diff(data, axis=1) ** 2, axis=1))
if np.any(complexity == 0):
    raise ValueError(
        "Some trajectories are constant, making standard CID undefined. "
        "Choose another outcome or explicitly handle constant trajectories "
        "before clustering."
    )

distances = clusterer.calculate_cid(data)

assert np.isfinite(distances).all(), "CID produced invalid distances."

clusterer.plot_dendrogram(distances)
plt.show()

clusters = clusterer.apply_agglomerative_clustering(
    distances,
    n_clusters=N_CLUSTERS,
)

cluster_experiments = experiments.copy()
cluster_experiments["clusters"] = clusters.astype(object)

fig, axes = plt.subplots(
    1, N_CLUSTERS,
    figsize=(5 * N_CLUSTERS, 4),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for c, ax in enumerate(axes[0]):
    selected = data[clusters == c]

    ax.plot(
        time,
        selected.T,
        color=plt.get_cmap("tab10")(c),
        alpha=0.15,
        linewidth=0.6,
    )

    ax.set(
        title=f"Cluster {c} ({len(selected)} runs)",
        xlabel="Model time",
    )

axes[0, 0].set_ylabel("Number of people")
fig.suptitle(OUTCOME_LABEL)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Green at the back, then blue, then orange on top
for zorder, c in enumerate([2, 0, 1], start=1):
    selected = data[clusters == c]

    ax.plot(
        time,
        selected.T,
        color=plt.get_cmap("tab10")(c),
        alpha=0.15,
        linewidth=0.6,
        zorder=zorder,
    )

# Clear legend with opaque colors
from matplotlib.lines import Line2D

handles = [
    Line2D(
        [0], [0],
        color=plt.get_cmap("tab10")(c),
        label=f"Cluster {c} ({np.sum(clusters == c)} runs)",
    )
    for c in range(N_CLUSTERS)
]

ax.legend(handles=handles)
ax.set(
    title=OUTCOME_LABEL,
    xlabel="Model time",
    ylabel="Number of people",
)

plt.tight_layout()
plt.show()

In [ ]:
display(
    pd.Series(clusters, name="Cluster")
    .value_counts()
    .sort_index()
    .rename("Number of runs")
)

# Explain cluster membership through input-space rules.
cluster_cart = cart.CART(
    X,
    clusters,
    mass_min=0.05,
    mode=RuleInductionType.CLASSIFICATION,
)

cluster_cart.build_tree()

In [ ]:

for number, (box, statistics) in enumerate(
    zip(cluster_cart.boxes, cluster_cart.stats),
    start=1,
):
    print(f"\nRule {number}")
    display(box)
    print(statistics)

**Student task:** Name each cluster using its trajectory, not its arbitrary number. Interpret two input rules. Repeat with 2 and 4 clusters and compare silhouette, interpretability, and test accuracy. Which findings survive changes in clustering choices?

## References and interpretation
- [EMA Workbench open exploration](https://emaworkbench.readthedocs.io/en/latest/indepth_tutorial/open-exploration.html)
- [SALib sampling](https://salib.readthedocs.io/en/latest/api/SALib.sample.html): structured Sobol designs.
- [SALib analysis](https://salib.readthedocs.io/en/latest/api/SALib.analyze.html): Sobol and PAWN definitions.
- [Scikit-learn tree rules](https://scikit-learn.org/stable/modules/generated/sklearn.tree.export_text.html).

Do not infer real-world probabilities from ensemble frequencies without justifying the input distributions. Scenario discovery here uses CART-style trees, not PRIM. A low held-out score is a substantive result: simple input rules may not describe the event well.
